# EOR-GRF SkyH5 Generation Workflow: Complete Code Trace

## From CLI to HDF5 Output — Unpacking the Black Box

This notebook provides a **detailed walkthrough** of how the `validation-sim` codebase generates EOR (Epoch of Reionization) sky models using **Gaussian Random Fields (GRF)**. We trace the complete pipeline:

```
CLI Arguments → Covariance Matrix → GRF Realization → SkyH5 Files
```

### What You'll Learn:
1. **Parameters**: `ell_max`, `nside`, `seed`, and their physical meaning
2. **Covariance Construction**: Power spectrum → C_ℓ → Pixel-space covariance
3. **Realization Sampling**: Cholesky decomposition and random field generation
4. **SkyModel Serialization**: pyradiosky.SkyModel → `.skyh5` HDF5 files
5. **Code Architecture**: Module structure, function calls, and data flow

---

**Repository Structure:**
```
validation-sim/
├── vsim.py                    # CLI entry point (Click framework)
└── core/
    ├── grf_covariance.py      # Covariance matrix computation
    ├── grf_realization.py     # Realization generation wrapper
    ├── sky_model.py           # SkyModel creation and serialization
    ├── eor_utils.py           # Power spectrum utilities
    ├── utils.py               # Constants, paths, frequencies
    └── slurm.py               # SLURM job submission decorator
```

In [ ]:
# ============================================================
# SECTION 1: Import Required Libraries and Setup
# ============================================================

from pathlib import Path
import numpy as np
import h5py
import sys

# Plotting
import matplotlib.pyplot as plt
try:
    import healpy as hp
    HAVE_HEALPY = True
except ImportError:
    HAVE_HEALPY = False
    print("healpy not available - some visualizations will be skipped")

# Astropy units
from astropy import units as u

# pyradiosky for SkyModel
try:
    from pyradiosky import SkyModel
    HAVE_PYRADIOSKY = True
except ImportError:
    HAVE_PYRADIOSKY = False
    print("pyradiosky not available")

# ============================================================
# Path Configuration
# ============================================================

VSIM_ROOT = Path("/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/validation-sim")
CORE_DIR = VSIM_ROOT / "core"
SKY_MODELS_DIR = VSIM_ROOT / "sky_models"

# Add core to path for importing internal modules
if str(CORE_DIR) not in sys.path:
    sys.path.insert(0, str(CORE_DIR))

print(f"validation-sim root: {VSIM_ROOT}")
print(f"Core modules at:     {CORE_DIR}")
print(f"Sky models at:       {SKY_MODELS_DIR}")

## Section 2: Overview — The Complete EOR-GRF Pipeline

### High-Level Workflow Diagram

```
┌─────────────────────────────────────────────────────────────────────────────┐
│                         EOR-GRF GENERATION PIPELINE                          │
└─────────────────────────────────────────────────────────────────────────────┘

    ┌──────────────────┐
    │  CLI ENTRY POINT │   vsim.py (Click framework)
    │    vsim.py       │   
    └────────┬─────────┘
             │
             ▼
    ┌──────────────────────────────────────────────────────────────────────┐
    │  STEP 1: COVARIANCE MATRIX                                            │
    │  ─────────────────────────────                                        │
    │  Command:  python vsim.py grf-covariance [--ell-max 1250]             │
    │  Module:   core/grf_covariance.py                                     │
    │  Function: compute_covariance()                                       │
    │                                                                        │
    │  Parameters:                                                           │
    │    • ell_max = 1250 (max angular multipole)                           │
    │    • k0 = logspace(-2, 1, 11) (k-modes in Mpc⁻¹)                      │
    │    • P(k) ∝ k^(-2.7) (EOR power-law model)                            │
    │                                                                        │
    │  Output: sky_models/raw/covariance.h5                                 │
    │          Shape: (N_freq, N_freq, ell_max+1)                           │
    └────────┬─────────────────────────────────────────────────────────────┘
             │
             ▼
    ┌──────────────────────────────────────────────────────────────────────┐
    │  STEP 2: GRF REALIZATION                                              │
    │  ────────────────────────                                             │
    │  Command:  python vsim.py grf-realization --nside 256 [--seed 2038]   │
    │  Module:   core/grf_realization.py                                    │
    │  External: calls `rgf` CLI from radiocosmology package                │
    │                                                                        │
    │  Parameters:                                                           │
    │    • nside = 256 (HEALPix resolution → npix = 12 × nside²)            │
    │    • seed = 2038 (random seed for reproducibility)                    │
    │    • low_memory = True (Cholesky vs full eigendecomposition)          │
    │                                                                        │
    │  Input:  sky_models/raw/covariance.h5                                 │
    │  Output: sky_models/raw/eor-grf-nside256.h5                           │
    │          Shape: (N_freq, 12×nside²) — healpix maps per freq           │
    └────────┬─────────────────────────────────────────────────────────────┘
             │
             ▼
    ┌──────────────────────────────────────────────────────────────────────┐
    │  STEP 3: SKYMODEL CREATION & SERIALIZATION                            │
    │  ──────────────────────────────────────────                           │
    │  Command:  python vsim.py sky-model grf-eor --channels 0-1023         │
    │  Module:   core/sky_model.py                                          │
    │  Function: make_grf_eor_model()                                       │
    │                                                                        │
    │  Wraps realization data in pyradiosky.SkyModel:                       │
    │    • component_type = "healpix"                                       │
    │    • stokes array shape: (4, 1, npix) — Jy/sr                        │
    │    • freq_array: single frequency per file                            │
    │                                                                        │
    │  Input:  sky_models/raw/eor-grf-nside256.h5                           │
    │  Output: sky_models/eor-grf-256/fch{0000..1023}.skyh5                 │
    └──────────────────────────────────────────────────────────────────────┘
```

### Key Dependencies

| Package | Purpose |
|---------|---------|
| `radiocosmology` | Power spectrum models, covariance computation, `rgf` CLI |
| `pyradiosky` | SkyModel container and `.skyh5` I/O |
| `astropy` | Cosmological parameters, units |
| `healpy` | HEALPix operations |
| `h5py` | HDF5 file I/O |

## Section 3: Understanding Sky Model Parameters

### Key Parameters and Their Physical Meaning

| Parameter | Default | Physical Meaning | Impact |
|-----------|---------|------------------|--------|
| `ell_max` | 1250 | Maximum angular multipole ℓ | Higher = finer angular resolution, more memory |
| `nside` | 256 | HEALPix resolution parameter | npix = 12 × nside² = 786,432 pixels for nside=256 |
| `seed` | 2038 | Random seed for GRF realization | Reproducibility |
| `k0` | logspace(-2, 1, 11) | Wavenumber modes (Mpc⁻¹) | Defines P(k) sampling |
| `P(k) ∝ k^α` | α = -2.7 | Power-law spectral index | EOR brightness fluctuation spectrum |

### Parameter Relationships

```
Angular Resolution:
    θ_min ≈ π / ell_max  (radians)
    
    ell_max = 1250  →  θ_min ≈ 0.14° ≈ 8.6 arcmin
    
HEALPix Resolution:
    npix = 12 × nside²
    pixel_area = 4π / npix  (steradians)
    
    nside = 256  →  npix = 786,432
                →  pixel_size ≈ 13.7 arcmin
    
Nyquist Constraint:
    ell_max should be ≲ 3 × nside for proper sampling
    nside = 256  →  ell_max ≲ 768 (conservative)
                →  ell_max = 1250 slightly oversampled
```

### Memory Scaling

```
Covariance Matrix:
    Size ∝ N_freq² × ell_max
    H6C: ~1024 frequencies, ell_max=1250
    → ~1.3 billion elements (10+ GB)
    
Realization Maps:
    Size ∝ N_freq × 12 × nside²
    → 1024 × 786,432 ≈ 805 million pixels
```

In [ ]:
# ============================================================
# Demonstrate Parameter Relationships
# ============================================================

def compute_healpix_params(nside):
    """Compute HEALPix parameters for a given nside."""
    npix = 12 * nside**2
    pixel_area_sr = 4 * np.pi / npix
    pixel_size_arcmin = np.sqrt(pixel_area_sr) * (180/np.pi) * 60
    ell_nyquist = 3 * nside  # Conservative Nyquist limit
    return {
        'nside': nside,
        'npix': npix,
        'pixel_area_sr': pixel_area_sr,
        'pixel_size_arcmin': pixel_size_arcmin,
        'ell_nyquist': ell_nyquist
    }

def compute_angular_resolution(ell_max):
    """Compute angular resolution for a given ell_max."""
    theta_min_rad = np.pi / ell_max
    theta_min_arcmin = theta_min_rad * (180/np.pi) * 60
    return {
        'ell_max': ell_max,
        'theta_min_rad': theta_min_rad,
        'theta_min_arcmin': theta_min_arcmin
    }

# Show typical parameter sets used in validation-sim
print("=" * 60)
print("HEALPIX RESOLUTION PARAMETERS")
print("=" * 60)
for nside in [64, 128, 256, 512]:
    p = compute_healpix_params(nside)
    print(f"\nnside = {p['nside']:4d}")
    print(f"  npix           = {p['npix']:,}")
    print(f"  pixel_size     = {p['pixel_size_arcmin']:.2f} arcmin")
    print(f"  ell_nyquist    = {p['ell_nyquist']}")

print("\n" + "=" * 60)
print("ANGULAR MULTIPOLE PARAMETERS")
print("=" * 60)
for ell_max in [256, 512, 768, 1250, 2000]:
    a = compute_angular_resolution(ell_max)
    print(f"\nell_max = {a['ell_max']:4d}")
    print(f"  θ_min = {a['theta_min_arcmin']:.2f} arcmin")

print("\n" + "=" * 60)
print("MEMORY ESTIMATES")
print("=" * 60)
n_freq = 1024  # H6C channels
ell_max = 1250

# Covariance: (n_freq, n_freq, ell_max+1) complex128
cov_size_bytes = n_freq * n_freq * (ell_max + 1) * 16  # 16 bytes per complex128
print(f"Covariance matrix (N_freq={n_freq}, ell_max={ell_max}):")
print(f"  Size = {cov_size_bytes / 1e9:.2f} GB")

# Realization: (n_freq, 12*nside^2) float64
for nside in [256, 512]:
    npix = 12 * nside**2
    real_size_bytes = n_freq * npix * 8  # 8 bytes per float64
    print(f"\nRealization maps (nside={nside}):")
    print(f"  Shape = ({n_freq}, {npix:,})")
    print(f"  Size  = {real_size_bytes / 1e9:.2f} GB")

## Section 4: Covariance Matrix Construction

### The Physics: From P(k) to C_ℓ

The EOR signal is modeled as a **Gaussian Random Field** characterized by its power spectrum. The pipeline converts a 3D power spectrum P(k) into angular power spectra C_ℓ(ν₁, ν₂) that describe correlations between sky pixels at different frequencies.

```
Physical Power Spectrum P(k):
    - k = wavenumber in Mpc⁻¹
    - P(k) ∝ k^(-2.7) for EOR-like fluctuations
    
Limber Approximation:
    C_ℓ(ν₁, ν₂) = ∫ P(k, z) × W(ν₁, z) × W(ν₂, z) dz
    
    where W(ν, z) is the window function relating frequency to redshift
```

### Code Location: `core/grf_covariance.py`

```python
# Key excerpt from grf_covariance.py (Lines 30-71)

def compute_covariance(ell_max=1250, test_mode=False, overwrite=False):
    """
    Compute the angular covariance matrix C_ℓ(ν₁, ν₂) for EOR GRF.
    
    Steps:
    1. Define power spectrum P(k) parameterization
    2. Set up frequency axis from H6C band
    3. Create covariance generator with cosmology
    4. Compute C_ℓ for all (ν₁, ν₂, ℓ) combinations
    5. Save to HDF5
    """
    
    # Power spectrum parameters (EOR model)
    k0 = np.logspace(-2., 1., 11)  # k-modes: 0.01 to 10 Mpc⁻¹
    a = k0**(-2.7)                 # Power-law amplitudes
    
    # Power spectrum model from radiocosmology
    from radiocosmology.power_spectrum import ParametricMultiPowerSpectrum
    ps = ParametricMultiPowerSpectrum(
        k=k0,
        a=a,
        term_type='flat_gauss',
        normalization_point=0.2,      # k_norm in Mpc⁻¹
        normalization_amplitude=1.0,  # P(k_norm)
    )
    
    # Frequency configuration (H6C band)
    freqs = utils.FREQS_DICT['H6C']  # Full H6C frequency array (Hz)
    nu_axis = 1e-6 * freqs           # Convert to MHz for covariance code
    
    # Angular multipole axis
    ell_axis = np.arange(0, ell_max + 1)  # 0, 1, 2, ..., ell_max
    
    # Covariance generator (uses astropy cosmology)
    from radiocosmology.covariance import CovarianceGenerator
    cov_gen = CovarianceGenerator(
        power_spectrum=ps,
        cosmology=astropy.cosmology.Planck18,
        Np=15,       # Integration points
        eps=1e-15,   # Numerical precision
    )
    
    # Compute covariance matrix
    C_ell = cov_gen.compute(nu_axis, ell_axis)
    # Shape: (N_freq, N_freq, N_ell)
    
    # Save to HDF5
    cov_gen.save('sky_models/raw/covariance.h5')
```

### Output File Structure: `covariance.h5`

```
covariance.h5
├── C_ell           # (N_freq, N_freq, N_ell) complex128
├── frequencies     # (N_freq,) float64 — in Hz
├── ell_axis        # (N_ell,) int64 — 0..ell_max
├── power_spectrum/ # Power spectrum parameters
│   ├── k0          # k-modes
│   ├── amplitudes  # P(k) values
│   └── term_type   # Parameterization type
└── cosmology/      # Cosmological parameters
    ├── H0
    ├── Om0
    └── ...
```

In [ ]:
# ============================================================
# Inspect the Covariance File (if it exists)
# ============================================================

cov_file = SKY_MODELS_DIR / "raw" / "covariance.h5"

if cov_file.exists():
    print(f"Inspecting: {cov_file}\n")
    
    with h5py.File(cov_file, 'r') as f:
        print("HDF5 File Structure:")
        print("=" * 50)
        
        def print_attrs(name, obj):
            print(f"  {name}")
            if isinstance(obj, h5py.Dataset):
                print(f"    shape: {obj.shape}, dtype: {obj.dtype}")
        
        f.visititems(print_attrs)
        
        # Try to read key arrays
        print("\n" + "=" * 50)
        print("Key Array Properties:")
        print("=" * 50)
        
        # Common dataset names to look for
        for key in ['C_ell', 'covariance', 'frequencies', 'freqs', 'ell', 'ell_axis']:
            if key in f:
                ds = f[key]
                print(f"\n{key}:")
                print(f"  shape: {ds.shape}")
                print(f"  dtype: {ds.dtype}")
                if ds.size < 100:
                    print(f"  values: {ds[:]}")
                else:
                    print(f"  first few: {ds[:5] if ds.ndim == 1 else ds[0, 0, :5]}")
else:
    print(f"Covariance file not found at: {cov_file}")
    print("This file is generated by: python vsim.py grf-covariance")
    print("\nShowing expected structure instead:")
    print("""
    covariance.h5 (expected):
    ├── C_ell or covariance  # (N_freq, N_freq, N_ell) — the main data
    ├── frequencies          # (N_freq,) — frequency axis in Hz
    ├── ell_axis             # (N_ell,) — multipole axis 0..ell_max
    └── metadata attrs       # Power spectrum params, cosmology
    """)

## Section 5: Realization Generation from Covariance

### The Mathematics: Gaussian Random Field Sampling

Given a covariance matrix **C**, we generate a realization **x** of a Gaussian random field:

```
Theory:
    x ~ N(0, C)  — multivariate normal with mean 0 and covariance C
    
Cholesky Method (low memory):
    1. Decompose: C = L × L^T  (Cholesky)
    2. Sample: z ~ N(0, I)     (iid standard normals)
    3. Transform: x = L × z    (correlated GRF)
    
Eigendecomposition Method (fast CPU):
    1. Decompose: C = U × Λ × U^T  (eigendecomposition)
    2. Sample: z ~ N(0, I)
    3. Transform: x = U × Λ^(1/2) × z
```

### Code Location: `core/grf_realization.py`

```python
# Key excerpt from grf_realization.py (Lines 18-22)

def generate_realization(nside, seed=2038, low_memory=True, overwrite=False):
    """
    Generate a GRF realization from the covariance matrix.
    
    This function wraps the external `rgf` CLI from the radiocosmology package.
    """
    
    # Paths
    cov_path = 'sky_models/raw/covariance.h5'
    out_path = f'sky_models/raw/eor-grf-nside{nside}.h5'
    
    # Build rgf command
    cmd = [
        'rgf', 'realization',
        '--nside', str(nside),
        '--seed', str(seed),
        '--low-memory' if low_memory else '--no-low-memory',
        '--covpath', cov_path,
        '--outpath', out_path,
    ]
    if overwrite:
        cmd.append('--overwrite')
    
    # Execute
    subprocess.run(cmd, check=True)
```

### The `rgf` CLI (from radiocosmology package)

```bash
# Command structure:
rgf realization \
    --nside 256 \
    --seed 2038 \
    --low-memory \
    --covpath sky_models/raw/covariance.h5 \
    --outpath sky_models/raw/eor-grf-nside256.h5 \
    --overwrite
```

**What `rgf realization` does internally:**
1. Reads covariance C_ℓ(ν₁, ν₂) from HDF5
2. For each multipole ℓ: constructs the (N_freq × N_freq) covariance block
3. Performs Cholesky decomposition (or eigendecomp)
4. Draws random spherical harmonics a_ℓm ~ N(0, C_ℓ)
5. Transforms to HEALPix pixel space via inverse SHT
6. Saves the (N_freq, N_pix) map to HDF5

### Output File Structure: `eor-grf-nside256.h5`

```
eor-grf-nside256.h5
├── healpix_maps    # (N_freq, 12*nside²) float64 — brightness in Jy/sr
├── frequencies     # (N_freq,) float64 — in Hz
├── nside           # scalar — HEALPix nside
├── seed            # scalar — random seed used
└── metadata/
    ├── ordering    # 'ring' or 'nested'
    └── units       # 'Jy/sr'
```

In [ ]:
# ============================================================
# Inspect the Realization File (if it exists)
# ============================================================

# Check for common nside values
for nside in [256, 512, 128]:
    real_file = SKY_MODELS_DIR / "raw" / f"eor-grf-nside{nside}.h5"
    
    if real_file.exists():
        print(f"Found realization file: {real_file}\n")
        
        with h5py.File(real_file, 'r') as f:
            print("HDF5 File Structure:")
            print("=" * 50)
            
            def print_structure(name, obj):
                if isinstance(obj, h5py.Dataset):
                    print(f"  {name}: shape={obj.shape}, dtype={obj.dtype}")
                elif isinstance(obj, h5py.Group):
                    print(f"  {name}/ (group)")
            
            f.visititems(print_structure)
            
            # Look for the main healpix maps
            for key in ['healpix_maps', 'maps', 'data']:
                if key in f:
                    maps = f[key]
                    print(f"\n{key} array:")
                    print(f"  shape: {maps.shape}")
                    print(f"  dtype: {maps.dtype}")
                    
                    # Compute some stats
                    if maps.shape[0] > 0:
                        # Read just one frequency slice to avoid memory issues
                        slice_0 = maps[0, :]
                        print(f"  First freq slice stats:")
                        print(f"    min:  {np.nanmin(slice_0):.4e}")
                        print(f"    max:  {np.nanmax(slice_0):.4e}")
                        print(f"    mean: {np.nanmean(slice_0):.4e}")
                        print(f"    std:  {np.nanstd(slice_0):.4e}")
                        print(f"    NaNs: {np.isnan(slice_0).sum()}")
                    break
            
            # Check for frequency array
            for key in ['frequencies', 'freqs', 'freq_array']:
                if key in f:
                    freqs = f[key][:]
                    print(f"\n{key}: {len(freqs)} channels")
                    print(f"  range: {freqs.min()/1e6:.2f} - {freqs.max()/1e6:.2f} MHz")
                    break
        
        break  # Found one, stop looking
else:
    print("No realization files found. Expected location:")
    print(f"  {SKY_MODELS_DIR / 'raw' / 'eor-grf-nside256.h5'}")
    print("\nGenerate with: python vsim.py grf-realization --nside 256")

## Section 6: Sky Model Object Creation and Serialization

### From Raw HEALPix Maps to pyradiosky SkyModel

The final step wraps the raw realization data into `pyradiosky.SkyModel` objects, one per frequency channel, and serializes them to `.skyh5` files.

### Code Location: `core/sky_model.py`

```python
# Key excerpt from sky_model.py (Lines 81-152)

def make_grf_eor_model(
    model_file: str,           # HDF5 file with healpix maps
    channels: list[int],       # Frequency channel indices
    label: str = "",           # Optional label for output directory
    offset_mode: str = "none", # "none", "constant", or "shift_min"
    offset_value: float = 0.0, # Offset for "constant" mode
    floor_epsilon: float = 1e-6,  # Floor for "shift_min" mode
):
    """
    Create pyradiosky SkyModel files from GRF realization.
    
    For each frequency channel:
    1. Read the HEALPix map from the raw realization file
    2. Apply optional offset (for handling negative pixels)
    3. Wrap in a pyradiosky.SkyModel object
    4. Write to .skyh5 format
    """
    
    # Read the raw realization
    with h5py.File(model_file, 'r') as f:
        healpix_maps = f['healpix_maps'][:]  # (N_freq, npix)
        nside = hp.npix2nside(healpix_maps.shape[1])
        freqs = f['frequencies'][:]  # Hz
    
    npix = 12 * nside**2
    
    # Output directory
    out_dir = Path(f'sky_models/eor-grf-{nside}{label}')
    out_dir.mkdir(parents=True, exist_ok=True)
    
    for ch in channels:
        # Extract single-frequency map
        sky_map = healpix_maps[ch, :]  # (npix,) in Jy/sr
        
        # Handle negative values (GRF can have negatives)
        if offset_mode == "constant":
            sky_map = sky_map + offset_value
        elif offset_mode == "shift_min":
            sky_map = sky_map - sky_map.min() + floor_epsilon
        
        # Build Stokes array: (4, 1, npix) — IQUV, 1 freq, npix
        # Only I (Stokes index 0) is non-zero
        stokes = np.zeros((4, 1, npix), dtype=np.float64) * u.Jy / u.sr
        stokes[0, 0, :] = sky_map * u.Jy / u.sr
        
        # Create SkyModel
        sky = SkyModel(
            component_type="healpix",
            nside=nside,
            hpx_order="ring",
            hpx_inds=np.arange(npix),
            spectral_type="full",
            freq_array=[freqs[ch]] * u.Hz,
            stokes=stokes,
            frame="icrs",
        )
        
        # Write to file
        out_path = out_dir / f"fch{ch:04d}.skyh5"
        sky.write_skyh5(str(out_path), clobber=True)
```

### SkyModel Object Structure

```python
SkyModel attributes:
    component_type = "healpix"     # Sky representation type
    nside = 256                    # HEALPix resolution
    hpx_order = "ring"             # Pixel ordering scheme
    hpx_inds = [0, 1, 2, ..., npix-1]  # All pixels
    spectral_type = "full"         # Full spectral info per channel
    freq_array = [freq_hz] * Hz    # Single frequency (one file per channel)
    stokes = Quantity((4, 1, npix), unit=Jy/sr)  # IQUV brightness
    frame = "icrs"                 # Coordinate frame
```

### Output File Structure: `fch0000.skyh5`

```
fch0000.skyh5 (HDF5)
├── Header/
│   ├── component_type    = "healpix"
│   ├── nside             = 256
│   ├── hpx_order         = "ring"
│   ├── spectral_type     = "full"
│   ├── frame             = "icrs"
│   └── freq_array        # (1,) — single frequency in Hz
├── Data/
│   ├── stokes            # (4, 1, npix) — Jy/sr
│   └── hpx_inds          # (npix,) — pixel indices
└── History/
    └── history           # provenance string
```

In [ ]:
# ============================================================
# Inspect a Generated SkyH5 File
# ============================================================

# Look for existing skyh5 files
eor_dirs = list(SKY_MODELS_DIR.glob("eor-grf-*"))
print(f"Found EOR-GRF output directories: {[d.name for d in eor_dirs]}\n")

skyh5_file = None
for eor_dir in eor_dirs:
    skyh5_files = list(eor_dir.glob("fch*.skyh5"))
    if skyh5_files:
        skyh5_file = skyh5_files[0]
        break

if skyh5_file and HAVE_PYRADIOSKY:
    print(f"Inspecting: {skyh5_file}\n")
    
    sky = SkyModel()
    sky.read_skyh5(str(skyh5_file))
    
    print("SkyModel Properties:")
    print("=" * 50)
    print(f"  component_type : {sky.component_type}")
    print(f"  nside          : {sky.nside}")
    print(f"  Ncomponents    : {sky.Ncomponents}")
    print(f"  spectral_type  : {sky.spectral_type}")
    print(f"  freq_array     : {sky.freq_array.to(u.MHz)}")
    
    print(f"\n  stokes shape   : {sky.stokes.shape}")
    print(f"  stokes unit    : {sky.stokes.unit}")
    
    # Stats on Stokes I
    I_map = sky.stokes[0, 0, :].to_value(u.Jy / u.sr)
    print(f"\n  Stokes I stats:")
    print(f"    min  : {np.nanmin(I_map):.4e} Jy/sr")
    print(f"    max  : {np.nanmax(I_map):.4e} Jy/sr")
    print(f"    mean : {np.nanmean(I_map):.4e} Jy/sr")
    print(f"    std  : {np.nanstd(I_map):.4e} Jy/sr")
    print(f"    NaNs : {np.isnan(I_map).sum()}")

elif skyh5_file:
    print(f"Found file: {skyh5_file}")
    print("But pyradiosky not available for inspection")
    
    # Fallback: direct HDF5 inspection
    with h5py.File(skyh5_file, 'r') as f:
        print("\nHDF5 structure:")
        def show(name, obj):
            if isinstance(obj, h5py.Dataset):
                print(f"  {name}: {obj.shape} {obj.dtype}")
        f.visititems(show)
else:
    print("No SkyH5 files found.")
    print("Generate with: python vsim.py sky-model grf-eor --channels 0-100 --nside 256")

## Section 7: CLI Argument Parsing and Entry Points

### The CLI Framework: Click

The `vsim.py` script uses the [Click](https://click.palletsprojects.com/) library to define a hierarchical command-line interface.

### CLI Structure in `vsim.py`

```python
# vsim.py structure (simplified)

import click

@click.group()
def cli():
    """Validation simulation pipeline."""
    pass

# ============================================================
# COVARIANCE COMMAND (Lines 305-318)
# ============================================================
@cli.command('grf-covariance')
@click.option('--test-mode/--production', default=False,
              help='Use reduced frequency/ell grid for testing')
@click.option('--ell-max', default=1250, type=int,
              help='Maximum angular multipole')
@click.option('--overwrite/--no-overwrite', default=False,
              help='Overwrite existing covariance file')
def grf_covariance(test_mode, ell_max, overwrite):
    """Compute the GRF covariance matrix."""
    from core.grf_covariance import compute_covariance
    compute_covariance(
        ell_max=ell_max,
        test_mode=test_mode,
        overwrite=overwrite,
    )

# ============================================================
# REALIZATION COMMAND (Lines 296-303)
# ============================================================
@cli.command('grf-realization')
@click.option('--nside', required=True, type=int,
              help='HEALPix nside parameter (e.g., 256, 512)')
@click.option('--seed', default=2038, type=int,
              help='Random seed for reproducibility')
@click.option('--low-memory/--fast-cpu', default=True,
              help='Use Cholesky (low-memory) vs eigendecomp (fast)')
@click.option('--overwrite/--no-overwrite', default=False)
def grf_realization(nside, seed, low_memory, overwrite):
    """Generate a GRF realization from covariance."""
    from core.grf_realization import generate_realization
    generate_realization(
        nside=nside,
        seed=seed,
        low_memory=low_memory,
        overwrite=overwrite,
    )

# ============================================================
# SKY-MODEL SUBGROUP (Lines 220-289)
# ============================================================
@cli.group('sky-model')
def sky_model():
    """Sky model generation commands."""
    pass

@sky_model.command('grf-eor')
@click.option('--nside', default=256, type=int,
              help='HEALPix resolution (must match realization)')
@click.option('--channels', required=True,
              help='Channel range, e.g., "0-1023" or "227-315"')
@click.option('--label', default='',
              help='Optional label for output directory')
@click.option('--offset-mode', default='none',
              type=click.Choice(['none', 'constant', 'shift_min']))
@click.option('--offset-value', default=0.0, type=float)
def grf_eor(nside, channels, label, offset_mode, offset_value):
    """Create SkyModel files from GRF realization."""
    from core.sky_model import make_grf_eor_model
    
    # Parse channel range (e.g., "0-1023" -> [0, 1, 2, ..., 1023])
    channels = parse_range(channels)
    
    model_file = f'sky_models/raw/eor-grf-nside{nside}.h5'
    make_grf_eor_model(
        model_file=model_file,
        channels=channels,
        label=label,
        offset_mode=offset_mode,
        offset_value=offset_value,
    )

if __name__ == '__main__':
    cli()
```

### Example CLI Invocations

```bash
# Step 1: Compute covariance matrix
python vsim.py grf-covariance --ell-max 1250 --production

# Step 2: Generate realization
python vsim.py grf-realization --nside 256 --seed 2038 --low-memory

# Step 3: Create SkyModel files
python vsim.py sky-model grf-eor --nside 256 --channels 0-1023

# Full pipeline with labels
python vsim.py sky-model grf-eor --nside 256 --channels 227-315 --label "_h6c"
# Output: sky_models/eor-grf-256_h6c/fch0227.skyh5 ... fch0315.skyh5
```

In [ ]:
# ============================================================
# Show the actual vsim.py CLI commands
# ============================================================

vsim_script = VSIM_ROOT / "vsim.py"

if vsim_script.exists():
    print(f"Reading CLI structure from: {vsim_script}\n")
    
    # Read and show relevant sections
    with open(vsim_script, 'r') as f:
        content = f.read()
    
    # Find Click decorators to identify commands
    import re
    
    # Find all @cli.command and @*.command decorators
    command_pattern = r"@\w+\.command\(['\"](\w+(?:-\w+)*)['\"]"
    commands = re.findall(command_pattern, content)
    
    print("Discovered CLI commands:")
    print("=" * 40)
    for cmd in sorted(set(commands)):
        print(f"  • {cmd}")
    
    # Find grf-related function definitions
    print("\n\nGRF-related functions in vsim.py:")
    print("=" * 40)
    
    grf_funcs = re.findall(r"def (grf_\w+|.*grf.*)\(([^)]*)\):", content)
    for func_name, args in grf_funcs:
        print(f"  def {func_name}({args})")
    
    # Show imports from core module
    print("\n\nImports from core module:")
    print("=" * 40)
    core_imports = re.findall(r"from core\.(\w+) import (\w+)", content)
    for module, func in core_imports:
        print(f"  from core.{module} import {func}")
else:
    print(f"vsim.py not found at: {vsim_script}")
    print("Showing expected CLI structure from documentation.")

## Section 8: Complete Workflow Trace (CLI to File Output)

### End-to-End Call Graph

```
┌────────────────────────────────────────────────────────────────────────────┐
│ USER COMMAND: python vsim.py grf-covariance --ell-max 1250                 │
└────────┬───────────────────────────────────────────────────────────────────┘
         │
         ▼
┌────────────────────────────────────────────────────────────────────────────┐
│ vsim.py                                                                     │
│   └── @cli.command('grf-covariance')                                       │
│       └── grf_covariance(test_mode, ell_max, overwrite)                    │
│           └── from core.grf_covariance import compute_covariance           │
└────────┬───────────────────────────────────────────────────────────────────┘
         │
         ▼
┌────────────────────────────────────────────────────────────────────────────┐
│ core/grf_covariance.py                                                      │
│   └── compute_covariance(ell_max=1250, test_mode=False, overwrite=False)   │
│       ├── k0 = np.logspace(-2., 1., 11)                                    │
│       ├── a = k0**(-2.7)                                                    │
│       │                                                                     │
│       ├── ParametricMultiPowerSpectrum(k=k0, a=a, ...)  ← radiocosmology   │
│       │   └── Returns: ps (power spectrum object)                          │
│       │                                                                     │
│       ├── freqs = utils.FREQS_DICT['H6C']               ← core/utils.py    │
│       ├── ell_axis = np.arange(0, ell_max+1)                               │
│       │                                                                     │
│       ├── CovarianceGenerator(power_spectrum=ps, ...)   ← radiocosmology   │
│       │   └── cov_gen.compute(nu_axis, ell_axis)                           │
│       │       └── Returns: C_ell array (N_freq, N_freq, N_ell)             │
│       │                                                                     │
│       └── cov_gen.save('sky_models/raw/covariance.h5')                     │
└────────────────────────────────────────────────────────────────────────────┘
                                    │
                                    ▼
                      📁 sky_models/raw/covariance.h5

════════════════════════════════════════════════════════════════════════════

┌────────────────────────────────────────────────────────────────────────────┐
│ USER COMMAND: python vsim.py grf-realization --nside 256 --seed 2038       │
└────────┬───────────────────────────────────────────────────────────────────┘
         │
         ▼
┌────────────────────────────────────────────────────────────────────────────┐
│ vsim.py                                                                     │
│   └── @cli.command('grf-realization')                                      │
│       └── grf_realization(nside, seed, low_memory, overwrite)              │
│           └── from core.grf_realization import generate_realization        │
└────────┬───────────────────────────────────────────────────────────────────┘
         │
         ▼
┌────────────────────────────────────────────────────────────────────────────┐
│ core/grf_realization.py                                                     │
│   └── generate_realization(nside=256, seed=2038, low_memory=True, ...)     │
│       │                                                                     │
│       └── subprocess.run(['rgf', 'realization', ...])   ← External CLI     │
│           │                                                                 │
│           └── rgf (from radiocosmology package)                            │
│               ├── Reads: covariance.h5                                     │
│               ├── Cholesky decomposition per ℓ                             │
│               ├── Samples a_ℓm ~ N(0, C_ℓ)                                 │
│               ├── Inverse spherical harmonic transform                     │
│               └── Writes: eor-grf-nside256.h5                              │
└────────────────────────────────────────────────────────────────────────────┘
                                    │
                                    ▼
                      📁 sky_models/raw/eor-grf-nside256.h5

════════════════════════════════════════════════════════════════════════════

┌────────────────────────────────────────────────────────────────────────────┐
│ USER COMMAND: python vsim.py sky-model grf-eor --nside 256 --channels 0-99 │
└────────┬───────────────────────────────────────────────────────────────────┘
         │
         ▼
┌────────────────────────────────────────────────────────────────────────────┐
│ vsim.py                                                                     │
│   └── @sky_model.command('grf-eor')                                        │
│       └── grf_eor(nside, channels, label, offset_mode, offset_value)       │
│           ├── channels = parse_range('0-99') → [0, 1, 2, ..., 99]          │
│           └── from core.sky_model import make_grf_eor_model                │
└────────┬───────────────────────────────────────────────────────────────────┘
         │
         ▼
┌────────────────────────────────────────────────────────────────────────────┐
│ core/sky_model.py                                                           │
│   └── make_grf_eor_model(model_file, channels, label, ...)                 │
│       │                                                                     │
│       ├── h5py.File(model_file) → reads healpix_maps, freqs                │
│       │                                                                     │
│       └── for ch in channels:                                              │
│           ├── sky_map = healpix_maps[ch, :]                                │
│           ├── Apply offset if needed                                        │
│           │                                                                 │
│           ├── stokes = np.zeros((4, 1, npix)) * u.Jy/u.sr                  │
│           ├── stokes[0, 0, :] = sky_map                                    │
│           │                                                                 │
│           ├── sky = SkyModel(                          ← pyradiosky        │
│           │       component_type="healpix",                                 │
│           │       nside=nside,                                              │
│           │       stokes=stokes,                                            │
│           │       freq_array=[freqs[ch]],                                   │
│           │       ...                                                       │
│           │   )                                                             │
│           │                                                                 │
│           └── sky.write_skyh5(f'fch{ch:04d}.skyh5')                        │
└────────────────────────────────────────────────────────────────────────────┘
                                    │
                                    ▼
                      📁 sky_models/eor-grf-256/fch0000.skyh5
                      📁 sky_models/eor-grf-256/fch0001.skyh5
                      ...
                      📁 sky_models/eor-grf-256/fch0099.skyh5
```

## Section 9: Inspecting Generated SkyH5 Files

### Verification Checklist

After generating SkyH5 files, verify:
1. **Shape consistency**: stokes array is (4, 1, 12×nside²)
2. **Frequency correctness**: freq_array matches H6C band
3. **Unit correctness**: stokes in Jy/sr
4. **No NaN/Inf values**
5. **Statistics match expectations** (mean ≈ 0 for GRF, std matches power spectrum amplitude)

In [ ]:
# ============================================================
# Comprehensive SkyH5 Inspection
# ============================================================

def inspect_skyh5_file(filepath, make_plot=True):
    """
    Inspect a SkyH5 file and optionally plot the HEALPix map.
    """
    filepath = Path(filepath)
    
    if not filepath.exists():
        print(f"File not found: {filepath}")
        return None
    
    print(f"Inspecting: {filepath.name}")
    print("=" * 60)
    
    if HAVE_PYRADIOSKY:
        sky = SkyModel()
        sky.read_skyh5(str(filepath))
        
        # Basic properties
        print(f"  component_type : {sky.component_type}")
        print(f"  nside          : {sky.nside}")
        print(f"  Ncomponents    : {sky.Ncomponents}")
        print(f"  spectral_type  : {sky.spectral_type}")
        
        freq_mhz = sky.freq_array.to(u.MHz).value[0]
        print(f"  frequency      : {freq_mhz:.3f} MHz")
        
        print(f"\n  stokes shape   : {sky.stokes.shape}")
        print(f"  stokes unit    : {sky.stokes.unit}")
        
        # Extract Stokes I
        I_map = sky.stokes[0, 0, :].to_value(u.Jy / u.sr)
        
        # Statistics
        print(f"\n  Stokes I Statistics:")
        print(f"    min    : {np.nanmin(I_map):.6e} Jy/sr")
        print(f"    max    : {np.nanmax(I_map):.6e} Jy/sr")
        print(f"    mean   : {np.nanmean(I_map):.6e} Jy/sr")
        print(f"    median : {np.nanmedian(I_map):.6e} Jy/sr")
        print(f"    std    : {np.nanstd(I_map):.6e} Jy/sr")
        print(f"    NaNs   : {np.isnan(I_map).sum()}")
        print(f"    Infs   : {np.isinf(I_map).sum()}")
        
        # Verify shape consistency
        expected_npix = 12 * sky.nside**2
        actual_npix = I_map.size
        if actual_npix == expected_npix:
            print(f"\n  ✓ npix matches: {actual_npix} = 12 × {sky.nside}²")
        else:
            print(f"\n  ✗ npix mismatch: {actual_npix} ≠ {expected_npix}")
        
        # Plot if healpy available
        if make_plot and HAVE_HEALPY:
            plt.figure(figsize=(10, 5))
            hp.mollview(
                I_map,
                title=f"{filepath.name} @ {freq_mhz:.2f} MHz",
                unit="Jy/sr",
                nest=False,  # ring ordering
            )
            plt.show()
            
            # Histogram
            plt.figure(figsize=(8, 4))
            plt.hist(I_map[np.isfinite(I_map)], bins=100, log=True, alpha=0.7)
            plt.xlabel("Brightness [Jy/sr]")
            plt.ylabel("Pixel count")
            plt.title(f"Pixel Distribution: {filepath.name}")
            plt.grid(alpha=0.3)
            plt.show()
        
        return sky
    
    else:
        # Fallback to raw HDF5
        with h5py.File(filepath, 'r') as f:
            print("HDF5 structure (pyradiosky not available):")
            def show(name, obj):
                if isinstance(obj, h5py.Dataset):
                    print(f"  {name}: {obj.shape} {obj.dtype}")
            f.visititems(show)
        return None

# Find and inspect a file
for eor_dir in sorted(SKY_MODELS_DIR.glob("eor-grf-*")):
    skyh5_files = sorted(eor_dir.glob("fch*.skyh5"))
    if skyh5_files:
        # Inspect first file
        inspect_skyh5_file(skyh5_files[0], make_plot=True)
        break
else:
    print("No EOR-GRF SkyH5 files found to inspect.")

## Section 10: Validating Covariance Properties in Realizations

### Validation Approaches

1. **Power Spectrum Recovery**: Compute C_ℓ from the generated map and compare to input
2. **Sample Covariance**: Generate multiple realizations and verify sample covariance matches theory
3. **Isotropy Check**: Verify rotational invariance of statistics
4. **Frequency Correlation**: Check that cross-frequency correlations match the covariance model

In [ ]:
# ============================================================
# Power Spectrum Recovery from Generated Maps
# ============================================================

def compute_angular_power_spectrum(healpix_map, lmax=None):
    """
    Compute the angular power spectrum C_ℓ from a HEALPix map.
    
    Uses healpy.anafast for spherical harmonic transform.
    """
    if not HAVE_HEALPY:
        print("healpy required for power spectrum computation")
        return None, None
    
    nside = hp.npix2nside(len(healpix_map))
    if lmax is None:
        lmax = 3 * nside - 1  # Nyquist limit
    
    # Compute power spectrum
    cl = hp.anafast(healpix_map, lmax=lmax)
    ell = np.arange(len(cl))
    
    return ell, cl

def validate_power_spectrum(skyh5_files, n_samples=5, lmax=500):
    """
    Load multiple SkyH5 files (different frequencies) and compute their
    angular power spectra for validation.
    """
    if not HAVE_PYRADIOSKY or not HAVE_HEALPY:
        print("Requires pyradiosky and healpy")
        return
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    freqs_mhz = []
    all_cls = []
    
    # Sample every Nth file
    step = max(1, len(skyh5_files) // n_samples)
    sampled_files = skyh5_files[::step][:n_samples]
    
    for skyfile in sampled_files:
        sky = SkyModel()
        sky.read_skyh5(str(skyfile))
        
        freq_mhz = sky.freq_array.to(u.MHz).value[0]
        freqs_mhz.append(freq_mhz)
        
        I_map = sky.stokes[0, 0, :].to_value(u.Jy / u.sr)
        
        ell, cl = compute_angular_power_spectrum(I_map, lmax=lmax)
        all_cls.append(cl)
        
        # Plot power spectrum
        axes[0].loglog(ell[2:], cl[2:], label=f"{freq_mhz:.1f} MHz", alpha=0.7)
    
    axes[0].set_xlabel(r"Multipole $\ell$")
    axes[0].set_ylabel(r"$C_\ell$ [(Jy/sr)$^2$]")
    axes[0].set_title("Angular Power Spectrum by Frequency")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.3)
    
    # Cross-frequency correlation at fixed ℓ
    if len(all_cls) > 1:
        ell_check = 100  # Check at ℓ=100
        cl_at_ell = [cl[ell_check] for cl in all_cls]
        
        axes[1].plot(freqs_mhz, cl_at_ell, 'o-', markersize=8)
        axes[1].set_xlabel("Frequency [MHz]")
        axes[1].set_ylabel(f"$C_{{{ell_check}}}$ [(Jy/sr)$^2$]")
        axes[1].set_title(f"Power at $\\ell$={ell_check} vs Frequency")
        axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return ell, all_cls, freqs_mhz

# Run validation if files exist
for eor_dir in sorted(SKY_MODELS_DIR.glob("eor-grf-*")):
    skyh5_files = sorted(eor_dir.glob("fch*.skyh5"))
    if len(skyh5_files) >= 5:
        print(f"Validating power spectrum from: {eor_dir.name}")
        print(f"Found {len(skyh5_files)} SkyH5 files\n")
        validate_power_spectrum(skyh5_files, n_samples=5, lmax=500)
        break
else:
    print("Insufficient SkyH5 files for power spectrum validation.")
    print("Generate files first with: python vsim.py sky-model grf-eor --channels 0-100")

## Summary: Key Takeaways

### The Three-Step EOR-GRF Pipeline

| Step | Command | Input | Output | Key Parameters |
|------|---------|-------|--------|----------------|
| **1. Covariance** | `grf-covariance` | P(k) model | `covariance.h5` | `ell_max=1250` |
| **2. Realization** | `grf-realization` | `covariance.h5` | `eor-grf-nside256.h5` | `nside=256`, `seed=2038` |
| **3. SkyModel** | `sky-model grf-eor` | `eor-grf-nside256.h5` | `fch{xxxx}.skyh5` | `channels`, `offset_mode` |

### Key Code Locations

| Function | File | Lines | Purpose |
|----------|------|-------|---------|
| `compute_covariance()` | `core/grf_covariance.py` | 30-71 | Build C_ℓ(ν₁,ν₂) |
| `generate_realization()` | `core/grf_realization.py` | 18-22 | Wrap `rgf` CLI |
| `make_grf_eor_model()` | `core/sky_model.py` | 81-152 | Create SkyModel files |

### External Dependencies

- **`radiocosmology`**: Provides `ParametricMultiPowerSpectrum`, `CovarianceGenerator`, and `rgf` CLI
- **`pyradiosky`**: Provides `SkyModel` class for .skyh5 I/O
- **`healpy`**: HEALPix operations and spherical harmonic transforms

### Parameter Summary

```python
# Covariance Parameters (physics)
ell_max = 1250              # Max angular multipole
k0 = logspace(-2, 1, 11)    # Wavenumbers (Mpc⁻¹)
P(k) ∝ k^(-2.7)             # Power-law model

# Realization Parameters (numerical)
nside = 256                 # HEALPix resolution (npix = 786,432)
seed = 2038                 # Random seed
low_memory = True           # Use Cholesky decomposition

# SkyModel Parameters (format)
component_type = "healpix"
hpx_order = "ring"
spectral_type = "full"
frame = "icrs"
```

---

**This notebook traced the complete path from CLI invocation to HDF5 output,
showing exactly how each parameter enters the pipeline and how data flows
through the covariance → realization → serialization stages.**

In [ ]:
# ============================================================
# Quick Reference: Read the Actual Source Code
# ============================================================

def show_source_excerpt(filepath, start_line, end_line, title=None):
    """Display a portion of a source file with syntax highlighting context."""
    filepath = Path(filepath)
    if not filepath.exists():
        print(f"File not found: {filepath}")
        return
    
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    if title:
        print(f"\n{'='*70}")
        print(f" {title}")
        print(f"{'='*70}")
    
    print(f"# File: {filepath.name} (lines {start_line}-{end_line})")
    print("-" * 70)
    
    for i, line in enumerate(lines[start_line-1:end_line], start=start_line):
        print(f"{i:4d} | {line.rstrip()}")
    
    print("-" * 70)

# Show key excerpts from the actual source files if they exist

# 1. grf_covariance.py
grf_cov_file = CORE_DIR / "grf_covariance.py"
if grf_cov_file.exists():
    show_source_excerpt(grf_cov_file, 1, 50, "grf_covariance.py - Covariance Computation")

# 2. grf_realization.py  
grf_real_file = CORE_DIR / "grf_realization.py"
if grf_real_file.exists():
    show_source_excerpt(grf_real_file, 1, 30, "grf_realization.py - Realization Generation")

# 3. sky_model.py
sky_model_file = CORE_DIR / "sky_model.py"
if sky_model_file.exists():
    show_source_excerpt(sky_model_file, 80, 130, "sky_model.py - SkyModel Creation")

# If none exist, show placeholder
if not any([grf_cov_file.exists(), grf_real_file.exists(), sky_model_file.exists()]):
    print("Core module files not found in expected location.")
    print(f"Expected at: {CORE_DIR}")
    print("\nListing available files:")
    if CORE_DIR.exists():
        for f in sorted(CORE_DIR.glob("*.py")):
            print(f"  {f.name}")

## Appendix: Complete CLI Option Reference

### Command: `grf-covariance`

Compute the angular covariance matrix C_ℓ(ν₁, ν₂) for EOR Gaussian Random Field.

```bash
python vsim.py grf-covariance [OPTIONS]
```

| Option | Type | Default | Description |
|--------|------|---------|-------------|
| `--test-mode` / `--production` | flag | `--production` | **Test mode**: Use reduced frequency grid (2 channels) and reduced ℓ grid (64 ℓ values) for quick testing. **Production**: Use full H6C band (~1024 channels) and full ℓ axis. |
| `--ell-max` | int | `1250` | Maximum angular multipole ℓ. Higher values capture finer angular structure but require more memory and compute time. Memory scales as O(N_freq² × ell_max). |
| `--local` / `--slurm` | flag | `--slurm` | Run locally in the current shell or submit as a SLURM job. SLURM recommended for production (requires ~48GB RAM, ~4 hours). |

**Example:**
```bash
# Quick test run
python vsim.py grf-covariance --test-mode --local

# Full production run on SLURM
python vsim.py grf-covariance --ell-max 1250 --production
```

---

### Command: `grf-realization`

Generate a GRF sky realization from the covariance matrix.

```bash
python vsim.py grf-realization [OPTIONS]
```

| Option | Type | Default | Description |
|--------|------|---------|-------------|
| `--nside` | int | **required** | HEALPix resolution parameter. Number of pixels = 12 × nside². Common values: 64 (test), 256 (standard), 512 (high-res). |
| `--seed` | int | `2038` | Random seed for reproducibility. Same seed + same covariance = identical realization. |
| `--low-memory` / `--fast-cpu` | flag | `--low-memory` | **Low-memory**: Use Cholesky decomposition (slower, less RAM). **Fast-cpu**: Use full eigendecomposition (faster, requires more RAM). |
| `--local` / `--slurm` | flag | `--slurm` | Run locally or submit to SLURM. High nside (≥512) requires large-memory nodes (RM-512 partition). |

**Example:**
```bash
# Standard resolution
python vsim.py grf-realization --nside 256 --seed 2038 --low-memory

# High resolution (needs big-memory node)
python vsim.py grf-realization --nside 512 --seed 42 --fast-cpu
```

---

### Command: `sky-model grf-eor`

Create pyradiosky SkyModel files from a GRF realization.

```bash
python vsim.py sky-model grf-eor [OPTIONS]
```

| Option | Type | Default | Description |
|--------|------|---------|-------------|
| `--nside` | int | `256` | HEALPix resolution. **Must match** the nside used in `grf-realization`. |
| `-fch` / `--channels` | range | `0~1024` | Frequency channels to process. Format: `227~315` or `0~100`. Can specify multiple ranges. |
| `--freq-range` | float float | `0 inf` | Alternative: specify frequency range in MHz instead of channel numbers. |
| `--label` | str | `""` | Label to append to output directory name. Output: `sky_models/eor-grf-{nside}{label}/` |
| `--local` / `--slurm` | flag | `--slurm` | Run locally or submit to SLURM. |
| `--split-freqs` / `--no-split-freqs` | flag | `--no-split-freqs` | If set, submit one SLURM job per frequency channel (for parallelism). |
| `--skip-existing` / `--rerun-existing` | flag | `--skip-existing` | Skip channels that already have output files. |
| `--dry-run` | flag | off | Print what would be done without actually running. |

**Internal parameters in `make_grf_eor_model()` (hardcoded):**

| Parameter | Value | Description |
|-----------|-------|-------------|
| `offset_mode` | `"shift_min"` | How to handle negative pixel values in GRF. Options: `"none"` (keep negatives), `"constant"` (add fixed offset), `"shift_min"` (shift so min=floor_epsilon). |
| `floor_epsilon` | `1e-6` | Minimum brightness floor when using `shift_min` mode. Prevents zero/negative values. |
| `offset_value` | `1e-5` | Fixed offset when using `offset_mode="constant"`. |

**Example:**
```bash
# Process H6C band channels
python vsim.py sky-model grf-eor --nside 256 --channels 227~315 --label "_h6c"

# Output directory: sky_models/eor-grf-256_h6c/
# Files: fch0227.skyh5, fch0228.skyh5, ..., fch0314.skyh5
```

---

### Command: `runsim` (Main Simulation Command)

Run visibility simulations using the generated sky models.

```bash
python vsim.py runsim [OPTIONS]
```

| Option | Type | Default | Description |
|--------|------|---------|-------------|
| `-a` / `--ants` | range | required | Antenna indices. Format: `0~10` or `0~350`. |
| `--layout` | choice | None | Pre-defined array layout (alternative to `--ants`). |
| `-fch` / `--channels` | range | all | Frequency channels. |
| `-sm` / `--sky-model` | choice | `ptsrc` | Sky model directory to use: `ptsrc`, `eor-grf-256`, etc. |
| `--n-time-chunks` | int | `3` | Number of time chunks to split simulation. |
| `--do-time-chunks` | range | all | Which time chunks to actually run. |
| `--simulator` | choice | `matvis` | Backend: `matvis`, `fftvis`, `matvis-cpu`, `fftvis32`, `fftvis64`. |
| `--beam-map-csv` | path | None | CSV file mapping antennas to beam files (matvis only). |
| `--beamvar-type` | choice | None | Beam variation type: `vivaldired`, `airyred`, `airyprb`, `airytilt`. |
| `--ideal-layout` / `--not-ideal-layout` | flag | `--ideal-layout` | Use idealized (redundant) antenna positions. |
| `--redundant` / `--not-redundant` | flag | `--not-redundant` | Only simulate redundant baselines. |
| `--spline-interp-order` | int | `3` | Spline interpolation order for beam. |
| `--skip-existing` / `--rerun-existing` | flag | `--skip-existing` | Skip existing output files. |

**Analytical Beam Options** (alternative to UVBeam files):

| Option | Type | Default | Description |
|--------|------|---------|-------------|
| `--analytic-beam-class` | choice | None | Beam class: `AiryBeam`, `GaussianBeam`, `hera_sim.beams.PolyBeam`, etc. |
| `--analytic-beam-diameter` | float | `14.0` | Dish diameter in meters (for AiryBeam). |
| `--analytic-beam-sigma` | float | `0.15` | Beam sigma in radians (for GaussianBeam). |
| `--analytic-beam-ref-freq` | float | `1.0e8` | Reference frequency in Hz. |
| `--analytic-beam-spectral-index` | float | `-0.6975` | Frequency scaling index. |
| `--analytic-beam-coeffs-file` | path | None | JSON/YAML file with beam coefficients. |
| `--analytic-beam-preset` | choice | None | Preset coefficients: `fagnoni19`. |
| `--analytic-beam-map-file` | path | None | YAML file mapping antennas to different beam configs. |

**Example:**
```bash
# Run EOR simulation with analytical Airy beam
python vsim.py runsim \
    --ants 0~10 \
    --channels 227~257 \
    --sky-model eor-grf-256 \
    --n-time-chunks 288 \
    --do-time-chunks 0~10 \
    --simulator matvis \
    --analytic-beam-class AiryBeam \
    --analytic-beam-diameter 14.0
```

---

### Quick Reference: Channel Range Syntax

The `--channels` option uses a special range syntax:

| Input | Result |
|-------|--------|
| `227` | Single channel 227 |
| `227~257` | Channels 227, 228, ..., 256 (257 exclusive) |
| `0~100` | Channels 0-99 |
| `--channels 0~50 --channels 100~150` | Channels 0-49 and 100-149 (multiple ranges) |

---

### Quick Reference: Offset Modes for GRF

GRF realizations can have negative brightness values. The `offset_mode` parameter controls how these are handled:

| Mode | Formula | Use Case |
|------|---------|----------|
| `"none"` | `I_out = I_grf` | Keep negatives (for analysis, not simulation) |
| `"constant"` | `I_out = I_grf + offset_value` | Add fixed offset (e.g., 1e-5 Jy/sr) |
| `"shift_min"` | `I_out = I_grf - min(I_grf) + floor_epsilon` | Shift so minimum = floor_epsilon |

**Current default in validation-sim**: `offset_mode="shift_min"`, `floor_epsilon=1e-6`

## Appendix: Log Files and Debugging

### Log File Locations

Each SLURM job writes its stdout/stderr to a log file. The log files are named with the SLURM job ID.

| Step | Command | Log Directory | File Pattern |
|------|---------|---------------|--------------|
| **1** | `grf-covariance` | `logs/grf-covariance/` | `<JOBID>.out` |
| **2** | `grf-realization` | `logs/grf-realization/` | `<JOBID>.out` |
| **3** | `sky-model grf-eor` | `logs/skymodel/grf-eor{nside}/` | `<JOBID>.out` |

### Full Paths

```bash
# Step 1: grf-covariance logs
/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/validation-sim/logs/grf-covariance/

# Step 2: grf-realization logs
/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/validation-sim/logs/grf-realization/

# Step 3: sky-model logs (nside determines subfolder)
/lustre/aoc/projects/hera/rchandra/H6C_Validation_Stats/validation-sim/logs/skymodel/grf-eor256/
```

### Output Data Products

| Step | Output Location | Description |
|------|-----------------|-------------|
| **1** | `sky_models/raw/covariance.h5` | Frequency-frequency covariance matrix C_ℓ(ν₁, ν₂) |
| **2** | `sky_models/raw/eor-grf-nside{nside}.h5` | HEALPix maps per frequency channel |
| **3** | `sky_models/grf-eor{nside}/fch{NNNN}.skyh5` | Individual SkyModel files per frequency |

### Sbatch Script Locations

The generated SLURM batch scripts are saved for inspection:

```bash
# grf-covariance sbatch
batch_scripts/grf-covariance/job.sbatch

# grf-realization sbatch
batch_scripts/grf-realization/job.sbatch

# sky-model sbatch
batch_scripts/skymodel/grf-eor_allfreqs.sbatch
# or per-frequency:
batch_scripts/skymodel/grf-eor_fch0227.sbatch
```

### Debugging Commands

```bash
# Check running/pending jobs
squeue -u $USER

# Find your job ID after submission
# Output: "Submitted batch job 1234567"

# Tail the log in real-time
tail -f logs/grf-covariance/<JOBID>.out
tail -f logs/grf-realization/<JOBID>.out
tail -f logs/skymodel/grf-eor256/<JOBID>.out

# Find the most recent log file
ls -lt logs/grf-covariance/ | head -2
ls -lt logs/grf-realization/ | head -2
ls -lt logs/skymodel/grf-eor256/ | head -2

# View the most recent log
cat logs/grf-covariance/$(ls -t logs/grf-covariance/ | head -1)
cat logs/grf-realization/$(ls -t logs/grf-realization/ | head -1)
cat logs/skymodel/grf-eor256/$(ls -t logs/skymodel/grf-eor256/ | head -1)

# Check job status by ID
scontrol show job <JOBID>

# Cancel a job
scancel <JOBID>
```

### Expected Runtime and Resources

Based on historical logs:

| Step | Command | Expected Runtime | Memory | Notes |
|------|---------|------------------|--------|-------|
| **1** | `grf-covariance --ell-max 1250` | ~2 hours | 48 GB | Scales as O(ell_max²) |
| **1** | `grf-covariance --ell-max 512` | ~40-50 min | 48 GB | Reduced ell_max |
| **2** | `grf-realization --nside 256` | ~20 min | 9 GB | Eigen-decomposition: ~2.5 min |
| **2** | `grf-realization --nside 512` | ~1-2 hours | 20+ GB | Requires RM-512 partition |
| **3** | `sky-model grf-eor` (90 channels) | ~1 min | 16 GB | I/O dominated, fast |

### Overwrite Behavior

All three steps will **overwrite** existing output files:

| Step | Overwrite Mechanism |
|------|---------------------|
| `grf-covariance` | Deletes existing `covariance.h5` before saving (patched in `core/grf_covariance.py`) |
| `grf-realization` | Has `--overwrite` flag hardcoded in the `rgf` command |
| `sky-model grf-eor` | Uses `clobber=True` in `write_skyh5()` |

**Note**: The original `radiocosmology` package's `save_covariance_data()` does NOT support overwriting — it raises a `ValueError` if the file exists. We patched `core/grf_covariance.py` to delete the existing file first.

No additional flags needed for overwriting.